Scattering

In [ ]:
import numpy as np
from scipy.special import spherical_jn as j, spherical_yn as n # add as j, and as n for later use
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.optimize import minimize_scalar, minimize

Things i changed:
-mass of kripton
-the constant for the small value of the distance

STEP 1

Step 1: 1.1-1.2

In [ ]:
#defining the parameters
mH = 1.00784 #u
mKr = 83.798 #u
mu = (mH*mKr)/(mH+mKr) #u
eps = 5.99 #meV
sigma = 2 #A

#calculating hReduced^2/2*mu
conv = ((197.3**2)/(2*931.49))/10
conv = conv/mu

#calculating 2*m/hReduced^2
conv = 1/conv
def convFunction(m1,m2):
  mu = (m1*m2)/(m1+m2)
  conv = ((197.3**2)/(2*931.49))/10
  conv = conv/mu
  return 1/conv

Step 1: 1.3-1.4

In [ ]:
p1 = 12
p2 = 6
def potential(x,eps,sigma):
  global p1,p2
  x = np.asarray(x)
  return 4*eps*((sigma/x)**p1-(sigma/x)**p2)

In [ ]:
#for this calculation see report
def ansatz_smallx(x,eps,sigma,conv):
    return np.exp(-(np.sqrt(4*eps*conv*sigma**12/25)/x**5))

In [ ]:
def shoot(startX,xMax,h,startU1,startU2,E,l,eps,sigma,conv):

    #initialize x
    x = np.arange(startX,xMax,h)
    # initialize u
    u = np.zeros(len(x))
    if startU1==None: startU1 = ansatz_smallx(x[0],eps,sigma,conv)
    u[0] = startU1
    if startU2==None: startU2 = ansatz_smallx(x[1],eps,sigma,conv)
    u[1] = startU2
    """
    u[0] = x[0]**(l + 1)  # small x behavior for l > 0
    u[1] = x[1]**(l + 1)  # small x behavior for l > 0
    """
    # Numerov iteration
    K = (E - potential(x,eps,sigma))*conv - l*(l+1)/(x**2)

    for i in range(1, len(x)-1):
        u[i+1] = (
            2*u[i]*(1 - 5*h**2*K[i]/12)
            - u[i-1]*(1 + h**2*K[i-1]/12)
        ) / (1 + h**2*K[i+1]/12)

    return x,u

Step 1: 1.5

Produce a serier of curves for different values of l, in the range 0...6, and verify that for r>>$\sigma$ the solution is oscillatory with a phase that depends on l.

In [ ]:
lToPlot = 6
startX = sigma/2
xMax = sigma*20
h = 0.01
secondMax = []
for i in range(lToPlot+1):
  xx,uu = shoot(sigma/2,xMax,h,None,None,1.5,i,eps,sigma,conv)
  norm = np.sum(uu**2) * h
  u_normalized = uu / np.sqrt(norm)
  secondMax.append((i,xx[find_peaks(u_normalized)[0][1]]))
  plt.plot(xx, u_normalized, label="L={}".format(i))

plt.xlabel("Radius (A)")
plt.ylabel("")
plt.legend()

The plot shows us that the function is oscillatory for r>>$\sigma$ and this show also that we have a change of phase dependent on l. Now we can plot the peaks position for the different values of l.

In [ ]:
plt.scatter([i[0] for i in secondMax],[i[1] for i in secondMax])
plt.xlabel("l")
plt.ylabel("Radius (A)")
plt.title("Distance of the second peaks of the radial wavefunction")
plt.savefig("phaseChangeForl.png",dpi=600)

STEP 2

Step 2: 2.1

Implement a routine that computer the Bessel functions $j_l$ and $n_l$.

In [ ]:
def bessel(l, s0, s1, x):
    return (2*l -1)* s1 / x - s0

def jCustom(l,x):
    if l == 0:
        return np.sin(x) / x
    elif l == 1:
        return np.sin(x) / x**2 - np.cos(x) / x
    else:
        s0 = j(0, x)
        s1 = j(1, x)
        for i in range(2, l+1):
            s2 = bessel(i, s0, s1, x)
            s0, s1 = s1, s2
        return s1

def nCustom(l,x):
    if l == 0:
        return -np.cos(x) / x
    elif l == 1:
        return -np.cos(x) / x**2 - np.sin(x) / x
    else:
        s0 = n(0, x)
        s1 = n(1, x)
        for i in range(2, l+1):
            s2 = bessel(i, s0, s1, x)
            s0, s1 = s1, s2
        return s1

x = np.linspace(1,40,100)
lToLog = 4
for l in [1, 2, 4, 8]:
  y1 = jCustom(l,x)
  y2 = nCustom(l,x)
  y3 = j(l,x)
  y4 = n(l,x)
  plt.plot(x, y1, label="j_{}(x)".format(l))
  plt.plot(x, y2, label="n_{}(x)".format(l))
  plt.plot(x, y3, label="spherical_jn_{}(x)".format(l), linestyle="dashed")
  plt.plot(x, y4, label="spherical_yn_{}(x)".format(l), linestyle="dashed")
  plt.xlabel("x")
  if l>=lToLog:
    plt.yscale("symlog")
    logP = " (log10)"
  else: logP = ""
  plt.ylabel("Bessel function{}".format(logP))
  plt.legend()
  plt.savefig("bassel{}.png".format(l),dpi=300)
  plt.show()

Step 2: 2.2

Compute the $\delta_l$ for the same energy used in step 1. Chose different value for the distance.

In [ ]:
def phaseshift(startX,xMax,h,startU1,startU2,E,l,eps,sigma,conv,s1=-100,s2=-30):
    xx,uu = shoot(startX,xMax,h,startU1,startU2,E,l,eps,sigma,conv)
    x1, x2 = xx[s1], xx[s2]
    u1, u2 = uu[s1], uu[s2]
    k = np.sqrt(E*conv)
    kappa = x2*u1 / (x1*u2)
    delta = np.arctan2( (j(l, k*x1) - kappa*j(l , k*x2)) , (n(l, k*x1) - kappa*n(l, k*x2)) ) # could use arctan2 for correct quadrant
    return delta

In [ ]:
sigma = 2
E = 1.5
startX = sigma/2
xMax = sigma*20
h = 0.005
lToPlot = 8

numChoices = 20
randomChoice = [tuple(np.sort(np.random.choice(range(-100,-2), size=2, replace=False))) for i in range(numChoices)]

phaseShifts = np.zeros((lToPlot + 1,numChoices))
for i in range(numChoices):
  for i2 in range(lToPlot + 1):
    phaseShifts[i2][i] = phaseshift(startX,xMax,h,None,None,E,i2,eps,sigma,conv,randomChoice[i][0],randomChoice[i][1])

for i in range(len(phaseShifts)): plt.scatter(range(len(phaseShifts[i])),phaseShifts[i]/np.pi, label="l = {} ({:.3f})".format(i,np.mean(phaseShifts[i]/np.pi)))

plt.gca().xaxis.set_visible(False)
plt.legend()
plt.title("Phase shift values for different point of sampling (E = 1.5 meV)")
plt.ylabel("$\delta_\ell$")
plt.xlabel("Index of random couple of point of sampling")
plt.savefig("phaseShiftSamplingg2.png",dpi=600)

STEP 3

Step 3: 3.1

Compute the phase shift for a range 0...3.5 meV.

In [ ]:
sigma = 2
startX = sigma/2
xMax = sigma*20
h = 0.001
lToPlot = [0,1,8]
ERange = np.linspace(0.01,eps,100)

for i in lToPlot:
  phaseShifts = []
  for i2 in ERange:
    phaseShift = phaseshift(startX,sigma*10,h,None,None,i2,i,eps,sigma,conv)
    phaseShifts.append(phaseShift)
  plt.plot(ERange,phaseShifts,label="l = {}".format(i))

plt.legend()
plt.ylabel("Phase shift")
plt.xlabel("Energy [meV]")
plt.savefig("phaseshiftdE.png",dpi=600)

Step 3: 3.2 3.3

Compute total cross section for different sets of l considerd.

In [ ]:
def total_cross_section(startX,xMax,h,startU1,startU2,E,lMax,eps,sigma,conv):
    k = np.sqrt(E*conv)
    L = np.arange(0, lMax+1)  # consider l=0 to lmax
    cs_total = 0
    for l in L:
      cs_l = (4*np.pi / k**2) * (2*l +1) * np.sin(phaseshift(startX,xMax,h,startU1,startU2,E,l,eps,sigma,conv))**2
      cs_total += cs_l
    return cs_total

In [ ]:
def totalCrossSectionFunction(lMax,ERange,startX,xMax,h,startU1,startU2,eps,sigma,conv):
  return ERange,np.array([total_cross_section(startX,xMax,h,startU1,startU2,i,lMax,eps,sigma,conv) for i in ERange])

In [ ]:
lTo = [2,4,6,8]

for i in lTo:
  ee1,cc1=totalCrossSectionFunction(i,ERange,startX,xMax,h,None,None,eps,sigma,conv)
  plt.plot(ee1,cc1,label="lMax = {}".format(i))

plt.legend()
plt.ylabel("Total cross section $\sigma_{tot} [\AA^2]$")
plt.xlabel("Energy [meV]")
plt.savefig("crosssectionforLhigh.png",dpi=300)

Plot each contribution of the different l to the total cross section $\sigma_{tot}$.

In [ ]:
sigma = 2
startX = sigma/2
xMax = sigma*20
h = 0.005
lToPlot = [6,7,8,9]
ERange = np.linspace(0.01,eps*2,100)

for i in lToPlot:
  phaseShifts = []
  for i2 in ERange:
    k = np.sqrt(i2*conv)
    phaseShift = phaseshift(startX,xMax,h,None,None,i2,i,eps,sigma,conv)
    phaseShift = (4*np.pi / k**2) * (2*i +1) * np.sin(phaseShift)**2
    phaseShifts.append(phaseShift)
  plt.plot(ERange,phaseShifts,label="l = {}".format(i))

plt.legend()
plt.ylabel("Cross section $\sigma_\ell\ [\AA]$")
plt.xlabel("Energy [meV]")
plt.savefig("phaseshiftdE3.png",dpi=300)

STEP 4

Step 4: 4.1

In [ ]:
E_exp = np.array([0.50, 1.59, 2.94])  # meV, experimental peaks from paper
dE_exp = np.array([0.02, 0.06, 0.12])  # meV, estimated experimental uncertainties

In [ ]:
def getPeaks(lMax,ERange,nPeaks,startX,xMax,h,startU1,startU2,eps,sigma,conv):
  xx,yy = totalCrossSectionFunction(lMax,ERange,startX,xMax,h,startU1,startU2,eps,sigma,conv)
  peaks,properties = find_peaks(yy)
  if len(peaks)>=nPeaks: peaks = peaks[:nPeaks]
  else:
    peaks = np.array([1000000000]*nPeaks)
    return peaks,peaks,peaks
  return peaks,xx[peaks],yy[peaks]

In [ ]:
def delta(sigma):
  global E_exp,eps,conv
  lMax = 8
  ERange = np.linspace(0.01,eps,200)
  startX = sigma/2
  xMax = sigma*20
  h = 0.005
  startU1 = None
  startU2 = None

  peaks,xPeaks,yPeaks = getPeaks(lMax,ERange,len(E_exp),startX,xMax,h,startU1,startU2,eps,sigma,conv)
  return np.sum((E_exp-xPeaks)**2)

In [ ]:
def deltaError(sigma):
  global E_exp,eps,conv,lMax,h,Er
  ERange = np.linspace(0.01,eps,Er)
  startX = sigma/2
  xMax = sigma*20
  startU1 = None
  startU2 = None

  peaks,xPeaks,yPeaks = getPeaks(lMax,ERange,startX,xMax,h,startU1,startU2,eps,sigma,conv)
  return np.sum((E_exp-xPeaks)**2)

In [ ]:
r = minimize_scalar(delta,bounds=(3,4),method="bounded")
sigmaOptima = r.x

In [ ]:
lMax = 8
h = 0.005/2
Er = 200
r1 = minimize_scalar(deltaError,bounds=(3,4),method="bounded")

In [ ]:
lMax = 9
h = 0.005
Er = 200
r2 = minimize_scalar(deltaError,bounds=(3,4),method="bounded")

In [ ]:
lMax = 8
h = 0.005
Er = 400
r3 = minimize_scalar(deltaError,bounds=(3,4),method="bounded")

Calculate the error.

In [ ]:
ee = r.x-np.array([r1.x,r2.x,r3.x])
ee = ee**2
ee = np.sum(ee)
ee = np.sqrt(ee)
print("Error: {}".format(ee))

Comparing found and theoretical value of $\sigma$.

In [ ]:
lMax = 11
ERange = np.linspace(eps/18,eps*1.4,400)
startX1 = sigmaOptima/2
sigmaE = 3.57
startX2 = sigmaE/2
xMax1 = sigmaOptima*10
xMax2 = sigmaE*10
h = 0.01
startU1 = None
startU2 = None

xx1,yy1 = totalCrossSectionFunction(lMax,ERange,startX1,xMax,h,startU1,startU2,eps,sigmaOptima,conv)
xx2,yy2 = totalCrossSectionFunction(lMax,ERange,startX2,xMax2,h,startU1,startU2,eps,sigmaE,conv)
xx1 = np.sqrt(xx1/mu)*439.2599
xx2 = np.sqrt(xx2/mu)*439.2599
plt.plot(xx1,yy1,label="sigma = {}".format(sigmaOptima))
plt.plot(xx2,yy2,label="sigma = {}".format(sigmaE))
plt.legend()
plt.ylabel("Total cross section $\sigma_{tot}\ [\AA^2]$")
plt.xlabel("Relative elocity [m/s]")
plt.savefig("plotComparison.png",dpi=300)

Using the measured value of total cross section to make better fit.

In [ ]:
def deltaExperiment(parameters):
  sigma = parameters[0]
  eps = parameters[1]
  potent = parameters[2]
  global EDataxx,EDatayy,conv,p1,p2
  lMax = 8
  ERange = np.linspace(0.01,eps,200)
  startX = sigma/2
  xMax = sigma*10
  h = 0.01
  startU1 = None
  startU2 = None
  p1 = 12
  p2 = potent
  yy = [total_cross_section(startX,xMax,h,startU1,startU2,i,lMax,eps,sigma,conv) for i in mu*(EDataxx/439.25)**2]
  yy = yy/np.max(yy)*3.725
  return np.sum((yy-EDatayy)**2)

In [ ]:
r5 = minimize(deltaExperiment,[3.5,5.9,6],bounds=[(3,4),(5.8,6),(5,7)],method="L-BFGS-B",options={"maxiter": 50})

In [ ]:
p1 = 12
p2 = 6
xx,yy = totalCrossSectionFunction(8,np.linspace(5.99/18,5.99,100),r.x/2,r.x*20,0.01,None,None,5.99,r.x,conv)
xx1 = np.sqrt(xx/mu)*439.25
yy1 = yy/np.max(yy)*4.87

In [ ]:
n = [29,49,48,49,39,49,49,49,49,49,49,62,36,49,49,49,43,53,49,21,49,49,49,47,44,45,49,21,49,32,39,49,34,36,36,36,26,36,36,49,69,42,49,49,49,49,47,36,36,21,36,27,26,31,34]

In [ ]:
s = [1.9,3.2,5.1,2.0,2.4,1.7,1.7,1.5,1.9,2.4,3.9,15.9,1.5,1.7,5.1,4.1,0.9,10.8,6.5,2.9,1.6,2.7,1.2,3.7,4.9,1.4,2.4,1.9,4.9,2.4,4.3,1.6,2.2,1.5,3.0,1.6,1.2,1.6,3.1,6.9,21.1,4.1,5.2,3.8,2.8,3.8,1.9,3.0,1.7,2.3,2.1,2.1,2.6,1.6,2.9,2.3,1.8,1.2,3.1,3.4,4.8,6.4,2.3,2.2,1.5,1.4,2.9,1.9,2.0,1.4,2.9]

In [ ]:
ts = [2.756,2.058,2.213,3.012,2.869,2.931,2.864,2.838,3.036,2.419,2.120,2.775,2.791,3.104,3.118,2.446,2.717,2.343,3.371,2.701,2.612,2.111,3.056,2.910,2.002,2.982,2.935,2.771,3.691,2.792,1.924,2.983,2.940,2.847,2.825,2.590,2.285,1.963,1.507,3.095,2.288,3.725,3.560,1.844,2.459,2.904,3.144,3.020,2.674,2.765,2.952,2.455,2.115,2.702,1.825,1.658,2.183,2.981,2.249,3.047,2.832,1.668,2.897,2.816,2.767,2.344,1.802,1.546,2.136,2.419,3.098]

In [ ]:
vv = [1172,602,487,717,834,948,1061,1176,775,660,546,315,886,738,431,465,1168,349,383,679,677,621,963,451,503,1000,1023,1172,404,1169,565,974,986,1088,1134,1283,1512,1739,2310,359,301,404,416,529,637,694,745,791,854,1165,916,1393,1620,1234,1905,2078,1608,1005,652,745,447,539,707,818,1237,1424,1891,2170,1609,1422,772]

In [ ]:
import pandas as pd
bbbb = pd.DataFrame([n,s,ts,vv]).T

In [ ]:
EData = (np.array(bbbb[2]),np.array(bbbb[3]))
d = 385
EDataxx = np.sqrt(EData[1]**2 + d**2 - 2*EData[1]*d*0.6947)
EDatayy = EData[0]*EData[1]/np.sqrt(EData[1]**2 + d**2 - 2*EData[1]*d*0.6947)
plt.scatter(EDataxx,EDatayy)
plt.plot(xx1,yy1)
plt.xlabel("Relative velocity [m/s]")
plt.ylabel("Total cross section [arb]")
plt.savefig("experimentComparison.png",dpi=300)

In [ ]:
np.max(EDatayy)